In [1]:
import csv, json, os, shutil

# ---- STUDENT-SPECIFIC VALUE ----
SEED_DIGIT = 0   # <- your SEED_DIGIT

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_CLASS = (SEED_DIGIT % 5) + 1

SRC = r"C:\Users\Nate\Lab6_File_Handling_Linsangan\Lab6_File_handling_Additional_Exercises\archive (1)\user_behavior_dataset.csv"
OUT_DIR = "output"
BACKUP_DIR = "backup"
EXPORTED = os.path.join(OUT_DIR, "selected_users.csv")
SUMMARY = os.path.join(OUT_DIR, "usage_summary.json")
BACKUP = os.path.join(BACKUP_DIR, "selected_users_backup.csv")

FIELDS = ["User ID", "Device Model", "Operating System",
          "Battery Drain (mAh/day)", "Data Usage (MB/day)"]

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

total = os_matched = 0
invalid = 0
selected = []
battery_sum = data_sum = 0.0

# Req 1: read the dataset, find records matching the OS
with open(SRC, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Columns:", reader.fieldnames)  # confirm names match FIELDS
    for row in reader:
        total += 1
        try:
            os_name = row["Operating System"].strip()
            cls = int(row["User Behavior Class"])
            battery = float(row["Battery Drain (mAh/day)"])
            data = float(row["Data Usage (MB/day)"])
        except (KeyError, ValueError, AttributeError):
            invalid += 1           # invalid or missing record
            continue
        if os_name != TARGET_OS:
            continue
        os_matched += 1
        # Req 2: select the target behavior class
        if cls == TARGET_CLASS:
            selected.append({k: row[k].strip() for k in FIELDS})
            battery_sum += battery
            data_sum += data

n_selected = len(selected)

# Req 3: averages, handling no matching records
if n_selected > 0:
    avg_battery = round(battery_sum / n_selected, 2)
    avg_data = round(data_sum / n_selected, 2)
else:
    avg_battery = avg_data = None
    print("No matching records found.")

# Req 4: export selected users
with open(EXPORTED, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader()
    w.writerows(selected)

# Req 5: verify exported row count, then write JSON summary
with open(EXPORTED, newline="", encoding="utf-8") as f:
    exported_rows = sum(1 for _ in csv.DictReader(f))
count_ok = exported_rows == n_selected

summary = {
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_CLASS,
    "total_records": total,
    "invalid_records": invalid,
    "records_matching_os": os_matched,
    "records_in_target_class": n_selected,
    "average_battery_drain_mAh_day": avg_battery,
    "average_data_usage_MB_day": avg_data,
    "exported_row_count": exported_rows,
    "count_verified": count_ok,
    "exported_csv": EXPORTED,
}
with open(SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Req 6: backup the exported CSV and report its size
shutil.copy2(EXPORTED, BACKUP)
backup_size = os.path.getsize(BACKUP)
summary["backup_file"] = BACKUP
summary["backup_size_bytes"] = backup_size
with open(SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Assessment table values
print("OS and behavior class:", TARGET_OS, "/ Class", TARGET_CLASS)
print("Total records in dataset:", total)
print("Records matching OS:", os_matched)
print("Records in target class:", n_selected)
print("Avg battery drain (mAh/day):", avg_battery)
print("Avg data usage (MB/day):", avg_data)
print("Count verification:", "MATCH" if count_ok else "MISMATCH",
      f"({exported_rows} exported vs {n_selected} selected)")
print("Exported CSV:", EXPORTED)
print("JSON summary:", SUMMARY)
print("Backup:", BACKUP, "-", backup_size, "bytes")

Columns: ['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']
OS and behavior class: Android / Class 1
Total records in dataset: 700
Records matching OS: 554
Records in target class: 114
Avg battery drain (mAh/day): 452.61
Avg data usage (MB/day): 202.49
Count verification: MATCH (114 exported vs 114 selected)
Exported CSV: output\selected_users.csv
JSON summary: output\usage_summary.json
Backup: backup\selected_users_backup.csv - 4091 bytes
